In [ ]:
print("Hello, World")

In [ ]:
!pip install nltk

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import nltk

# Download necessary datasets for NLP processing
nltk.download('stopwords')
nltk.download('punkt')

print("Setup complete!")

In [ ]:
df_csv = pd.read_csv(r"C:\Users\marty\OneDrive\Desktop\Electric_Vehicle_Population_Data.csv")
print(df_csv.head())

## 📊 Step 1: Load and Preview Dataset

### ✅ Dataset Loaded Successfully

I loaded the dataset **Electric_Vehicle_Population_Data.csv** using `pandas.read_csv()` and displayed the first 5 rows using `df.head()` to understand the structure of the data.

### 📌 Key Observations from Preview

- The dataset contains vehicle-level information.
- Important columns include:
  - **VIN (1-10)** – Partial vehicle identification number
  - **County, City, State, Postal Code**
  - **Model Year**
  - **Make and Model**
  - **Electric Vehicle Type** (BEV or PHEV)
  - **CAFV Eligibility**
  - **Electric Range**
  - **Legislative District**
  - **Vehicle Location (Latitude & Longitude)**
  - **Electric Utility**
  - **2020 Census Tract**

### 🔎 Initial Insights

- The dataset includes both:
  - **Battery Electric Vehicles (BEV)**
  - **Plug-in Hybrid Electric Vehicles (PHEV)**
- Electric range varies significantly (e.g., 21 miles to 215 miles in the preview).
- Geographic information is available, which allows for potential spatial analysis.
- Utility provider information can be used for regional energy analysis.

### ➡️ Next Step

I will now:
- Check dataset shape
- Inspect data types
- Identify missing values
- Perform initial exploratory data analysis (EDA)


In [ ]:
# Step 2.1: Dataset Overview
# This step helps me understand the basic structure and quality of the dataset before analysis.

# Display dataset info
df_csv.info()

# Display dataset shape (rows, columns)
print("Dataset Shape:", df_csv.shape)

# Check for missing values
print("\nMissing Values per Column:")
print(df_csv.isnull().sum())

# Display column names
print("\nColumn Names:")
print(df_csv.columns.tolist())

# Step 2.1 – Dataset Overview

## Objective
In this step, I examined the structure and quality of the dataset before performing deeper analysis.  
This helps me understand:

- The size of the dataset  
- The number of features  
- Data types of each column  
- The presence of missing values  

---

## Dataset Shape

- **Rows:** 276,828  
- **Columns:** 16  

This confirms that the dataset is large and suitable for robust statistical and machine learning analysis.

---

## Data Types Summary

The dataset contains:

- **Float64 columns:** 4  
- **Int64 columns:** 2  
- **Object (categorical/text) columns:** 10  

This indicates a mix of numerical and categorical features, which is ideal for both regression and classification modeling.

---

## Missing Values Analysis

Most columns contain complete data. However, the following columns contain missing values:

- **County:** 15  
- **City:** 15  
- **Postal Code:** 15  
- **Electric Range:** 8  
- **Legislative District:** 687  
- **Vehicle Location:** 99  
- **Electric Utility:** 15  
- **2020 Census Tract:** 15  

The majority of missing values are very small relative to the dataset size (276k+ rows), except for **Legislative District**, which may require special attention during data cleaning.

---

## Column Names

1. VIN (1-10)  
2. County  
3. City  
4. State  
5. Postal Code  
6. Model Year  
7. Make  
8. Model  
9. Electric Vehicle Type  
10. Clean Alternative Fuel Vehicle (CAFV) Eligibility  
11. Electric Range  
12. Legislative District  
13. DOL Vehicle ID  
14. Vehicle Location  
15. Electric Utility  
16. 2020 Census Tract  

---

## Initial Observations

- The dataset is large and well-structured.
- Missing values are minimal overall.
- The dataset contains geographic, vehicle specification, and regulatory information.
- It is suitable for:
  - Exploratory Data Analysis (EDA)
  - Predictive modeling
  - Market trend analysis
  - Policy and adoption analysis

---

## Next Step

Proceed to:

**Step 2.2 – Data Cleaning & Preprocessing**
- Handle missing values
- Correct data types where necessary
- Remove duplicates (if any)
- Standardize categorical variables


In [ ]:
# Step 3: Check for missing values
missing_values = df_csv.isnull().sum().sort_values(ascending=False)

# Display top missing columns
print("Missing Values per Column:\n")
print(missing_values)

# Optional: show percentage of missing values for better context
missing_percent = (df_csv.isnull().sum() / len(df_csv)) * 100
print("\nPercentage of Missing Values:\n")
print(missing_percent.sort_values(ascending=False))


## Step 3: Missing Value Analysis

In this step, I analyzed the dataset to identify missing values across all columns.  
Understanding missing data is essential before proceeding to cleaning, feature engineering, or modeling.

---

### 3.1 Missing Values Count (Sorted)

I calculated the total number of missing values per column and sorted them in descending order.

**Key Findings:**

- **Legislative District** → 687 missing values  
- **Vehicle Location** → 99 missing values  
- **County, City, Postal Code, Electric Utility, 2020 Census Tract** → 15 missing values each  
- **Electric Range** → 8 missing values  
- All remaining columns have **0 missing values**

This indicates that the dataset is largely complete, with only a few columns requiring attention.

---

### 3.2 Percentage of Missing Values

To better understand the impact of missing data relative to dataset size (276,828 records), I calculated the percentage of missing values.

**Highest Missing Percentages:**

- **Legislative District** → ~0.25%
- **Vehicle Location** → ~0.036%
- Other columns → Less than 0.01%

All missing values are **well below 1%**, which suggests:

- No severe data quality issue
- Missing data can be handled safely without major distortion
- Options include imputation or selective removal

---

### 3.3 Interpretation

The dataset is very clean overall.  
The most notable missing column is *Legislative District*, but even that represents only a small fraction of the total records.

Since missing percentages are minimal:
- We can consider imputing where appropriate
- Or dropping rows only if the feature becomes critical for modeling

---

### Next Step

Proceed to:
- Handle missing values appropriately (imputation or removal)
- Check for duplicate records
- Begin deeper exploratory data analysis (EDA)


In [ ]:
# =========================
# Step 4: Handle Missing Values
# =========================

# 1) Quick check (before)
print("Missing values BEFORE handling:\n")
print(df_csv.isnull().sum().sort_values(ascending=False))

# 2) Split columns by type
num_cols = df_csv.select_dtypes(include=["int64", "float64"]).columns
cat_cols = df_csv.select_dtypes(include=["object"]).columns

# 3) Fill numeric missing values with MEDIAN
for col in num_cols:
    if df_csv[col].isnull().sum() > 0:
        df_csv[col] = df_csv[col].fillna(df_csv[col].median())

# 4) Fill categorical missing values with MODE (most frequent)
for col in cat_cols:
    if df_csv[col].isnull().sum() > 0:
        mode_value = df_csv[col].mode(dropna=True)
        if len(mode_value) > 0:
            df_csv[col] = df_csv[col].fillna(mode_value[0])
        else:
            df_csv[col] = df_csv[col].fillna("Unknown")

# 5) Quick check (after)
print("\nMissing values AFTER handling:\n")
print(df_csv.isnull().sum().sort_values(ascending=False))


## Step 3: Missing Value Handling

In this step, I handled missing values identified during the dataset inspection phase.

### Missing Values Before Handling

The following columns contained missing values:

- **Legislative District** → 687 missing (~0.25%)
- **Vehicle Location** → 99 missing
- **County, City, Electric Utility, 2020 Census Tract, Postal Code** → 15 missing each
- **Electric Range** → 8 missing
- All other columns → 0 missing values

Although the percentage of missing data is relatively small, handling them ensures data consistency and prevents issues during modeling.

---

## Strategy Used

### 1️⃣ Numerical Columns
For numerical features:
- Missing values were filled using the **median**.
- The median is robust to outliers and maintains the distribution of the data.

### 2️⃣ Categorical Columns
For categorical features:
- Missing values were filled using the **mode** (most frequent value).
- If mode was not available, a fallback value `"Unknown"` was used.

This approach preserves dataset size while maintaining statistical integrity.

---

## Missing Values After Handling

After applying the imputation strategy:

- All columns now contain **0 missing values**.
- The dataset is clean and ready for:
  - Outlier detection
  - Feature engineering
  - Exploratory Data Analysis (EDA)
  - Model building

---

✅ Data Integrity Status: CLEAN  
📊 Dataset Size: 276,828 rows × 16 columns  


In [ ]:
# Step 5: Detect and handle duplicates

# Check for duplicate records
duplicate_count = df_csv.duplicated().sum()
print(f"Number of duplicate rows before removal: {duplicate_count}")

# Display a few duplicate rows (if any)
if duplicate_count > 0:
    display(df[df_csv.duplicated()].head())

# Remove duplicate rows
df = df_csv.drop_duplicates()

# Verify duplicates are removed
print(f"Number of duplicate rows after removal: {df_csv.duplicated().sum()}")
print(f"Total remaining rows in dataset: {len(df_csv)}")

## Step 5: Duplicate Detection and Removal

In this step, I checked the dataset for duplicate records to ensure data quality and prevent biased analysis or model distortion.

---

### Duplicate Check (Before Removal)

- Number of duplicate rows detected: **0**
- This indicates that the dataset does not contain repeated records.

Since duplicates can inflate model performance or distort statistical summaries, verifying this step is critical in any professional data workflow.

---

### Duplicate Removal Process

Although no duplicates were found, I still applied:

- `drop_duplicates()` as a safety step
- Re-checked the dataset to confirm integrity

---

### Verification Results

- Duplicate rows after removal: **0**
- Total remaining rows: **276,828**
- Total columns: **16**

---

## Conclusion

The dataset contains **no duplicate records**.

✅ Data Integrity Status: CLEAN  
📊 Dataset Size: 276,828 rows × 16 columns  

The dataset is now fully prepared for the next stage of analysis.


In [ ]:
# Step 2.5: Identify categorical and numerical columns

# Display all column names
print("All Columns:\n", df.columns.tolist(), "\n")

# Identify numerical columns (int or float types)
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print("Numerical Columns:\n", num_cols, "\n")

# Identify categorical columns (object or string types)
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
print("Categorical Columns:\n", cat_cols, "\n")

# Count of each type
print(f"Total Numerical Columns: {len(num_cols)}")
print(f"Total Categorical Columns: {len(cat_cols)}")

## Step 2.5: Identify Numerical and Categorical Columns

In this step, I classified the dataset features into numerical and categorical variables.  
This is essential for:

- Proper missing value handling  
- Outlier detection  
- Feature engineering  
- Model preparation (encoding & scaling)

---

### All Columns in the Dataset

The dataset contains the following 16 columns:

VIN (1-10), County, City, State, Postal Code, Model Year, Make, Model,  
Electric Vehicle Type, Clean Alternative Fuel Vehicle (CAFV) Eligibility,  
Electric Range, Legislative District, DOL Vehicle ID, Vehicle Location,  
Electric Utility, 2020 Census Tract

---

### Numerical Columns (6)

These columns contain numeric values (int64 or float64):

- Postal Code  
- Model Year  
- Electric Range  
- Legislative District  
- DOL Vehicle ID  
- 2020 Census Tract  

Total Numerical Columns: **6**

These variables will later be used for:
- Statistical summaries  
- Outlier detection  
- Scaling (if needed)  
- Model input features  

---

### Categorical Columns (10)

These columns contain object/string values:

- VIN (1-10)  
- County  
- City  
- State  
- Make  
- Model  
- Electric Vehicle Type  
- Clean Alternative Fuel Vehicle (CAFV) Eligibility  
- Vehicle Location  
- Electric Utility  

Total Categorical Columns: **10**

These variables will later require:
- Standardization (if necessary)  
- Encoding (Label or One-Hot Encoding) for machine learning  

---

## Structural Summary

- Dataset Size: **276,828 rows × 16 columns**
- Numerical Features: **6**
- Categorical Features: **10**

The dataset structure is now clearly defined and ready for the next analytical steps.


In [ ]:
# Step 2.6: Enforce Correct Data Types

print("Data types BEFORE enforcement:\n")
print(df_csv.dtypes)
print("\n" + "-"*50 + "\n")

# 1️⃣ Convert Postal Code to string (should NOT be numeric)
df_csv["Postal Code"] = df_csv["Postal Code"].astype("Int64").astype(str)

# 2️⃣ Convert Legislative District to nullable integer
df_csv["Legislative District"] = df_csv["Legislative District"].astype("Int64")

# 3️⃣ Convert 2020 Census Tract to string (it's an identifier, not a number)
df_csv["2020 Census Tract"] = df_csv["2020 Census Tract"].astype("Int64").astype(str)

# 4️⃣ Ensure Model Year is integer
df_csv["Model Year"] = df_csv["Model Year"].astype("Int64")

# 5️⃣ Ensure Electric Range is numeric
df_csv["Electric Range"] = pd.to_numeric(df_csv["Electric Range"], errors="coerce")

# 6️⃣ Ensure DOL Vehicle ID is integer (ID column)
df_csv["DOL Vehicle ID"] = df_csv["DOL Vehicle ID"].astype("Int64")

print("Data types AFTER enforcement:\n")
print(df_csv.dtypes)


## Step 2.6 — Enforce Correct Data Types

### Objective
In this step, I enforced the correct data types for each column to ensure consistency, prevent calculation errors, and prepare the dataset for advanced analysis and modeling.

---

### Code Used

```python
# Convert 2020 Census Tract to string (it's an identifier, not a number)
df_csv["2020 Census Tract"] = df_csv["2020 Census Tract"].astype("Int64").astype(str)

# Ensure Model Year is integer
df_csv["Model Year"] = df_csv["Model Year"].astype("Int64")

# Ensure Electric Range is numeric
df_csv["Electric Range"] = pd.to_numeric(df_csv["Electric Range"], errors="coerce")

# Ensure DOL Vehicle ID is integer (ID column)
df_csv["DOL Vehicle ID"] = df_csv["DOL Vehicle ID"].astype("Int64")

# Convert Postal Code to string (ZIP codes are identifiers)
df_csv["Postal Code"] = df_csv["Postal Code"].astype("Int64").astype(str)

print("Data types AFTER enforcement:\n")
print(df_csv.dtypes)
```

---

### Output Summary

After enforcement:

- Identifier columns (`2020 Census Tract`, `Postal Code`, `DOL Vehicle ID`) are correctly formatted.
- Numerical columns (`Model Year`, `Electric Range`) are stored as numeric types.
- Categorical columns remain as `object`.

Dataset size remains:

**276,828 rows × 16 columns**

---

### Observations

- No structural changes to row count.
- No unintended type conversions.
- All identifiers are now protected from numeric manipulation errors.
- Dataset structure is now consistent and analysis-ready.

---

### Why This Step Is Important

Enforcing correct data types:

- Prevents modeling errors
- Protects identifier columns from aggregation mistakes
- Ensures numeric operations work correctly
- Improves memory efficiency
- Makes the dataset production-ready

---

### Data Integrity Status

Data Type Consistency: ✅ CLEAN  
Missing Values: ✅ Handled  
Duplicates: ✅ None detected  



In [ ]:
# Step 6.4: Standardize column names

# Standardize all column names to lowercase and replace spaces or special characters with underscores
df.columns = (
    df.columns
    .str.strip()                 # remove leading/trailing spaces
    .str.lower()                 # convert to lowercase
    .str.replace(' ', '_')       # replace spaces with underscores
    .str.replace('/', '_')       # replace slashes if any
    .str.replace('-', '_')       # replace hyphens if any
)

# Verify new standardized column names
print("Standardized Column Names:\n")
print(df.columns.tolist())

## Step 6.4 — Standardize Column Names

### Objective  
In this step, I standardized all column names to ensure consistency, readability, and compatibility with Python operations and machine learning workflows.

---

### Why This Is Necessary

Raw datasets often contain:
- Uppercase and lowercase inconsistencies
- Spaces in column names
- Special characters (/, -, etc.)
- Leading or trailing whitespace

These can cause:
- Errors in feature selection
- Inconvenience during modeling
- Issues when exporting to production systems

To prevent this, I standardized all column names.

---

### Code Used

```python
# Standardize all column names
df.columns = (
    df.columns
    .str.strip()           # Remove leading/trailing spaces
    .str.lower()           # Convert to lowercase
    .str.replace(' ', '_') # Replace spaces with underscores
    .str.replace('/', '_') # Replace slashes
    .str.replace('-', '_') # Replace hyphens
)

# Verify changes
print("Standardized Column Names:\n")
print(df.columns.tolist())
```

---

### Output Summary

Standardized column names:

```
['vin_(1_10)', 'county', 'city', 'state', 'postal_code', 
 'model_year', 'make', 'model', 'electric_vehicle_type', 
 'clean_alternative_fuel_vehicle_(cafv)_eligibility', 
 'electric_range', 'legislative_district', 'dol_vehicle_id', 
 'vehicle_location', 'electric_utility', '2020_census_tract']
```

---

### Observations

- All column names are now lowercase.
- Spaces have been replaced with underscores.
- Special characters have been removed or standardized.
- Naming convention is now consistent and professional.

---

### Data Integrity Status

Data Type Consistency: ✅ CLEAN  
Missing Values: ✅ Handled  
Duplicates: ✅ None detected  
Column Naming Convention: ✅ Standardized  

---

### Why This Step Is Important

Standardized column names:

- Improve code readability
- Prevent syntax errors
- Make feature engineering easier
- Ensure smoother model building
- Improve deployment compatibility


In [ ]:
# Step 2.8: Save the Clean and Standardized Dataset
# This step exports the cleaned dataframe to a new CSV file for future use in EDA and modeling.

# Define file name for the cleaned dataset
cleaned_file_path = "Electric_Vehicle_Population_Data.csv"

# Save dataset to CSV format without index column
df.to_csv(cleaned_file_path, index=False)

# Confirm the save operation
print(f"✅ Cleaned dataset successfully saved as: {cleaned_file_path}")

## Step 2.8 — Save the Clean and Standardized Dataset

### Objective  
In this step, I exported the fully cleaned and standardized dataset into a new CSV file.  
This ensures I have a reusable, production-ready dataset for Exploratory Data Analysis (EDA) and Machine Learning.

---

### Why This Step Is Necessary

After completing:

- Missing value handling  
- Data type enforcement  
- Duplicate removal  
- Column name standardization  

It is important to save the cleaned version so that:

- I avoid repeating cleaning steps  
- I maintain a structured workflow  
- Future analysis starts from a reliable dataset  
- The project remains reproducible  

---

### Code Used

```python
# Define file name for cleaned dataset
cleaned_file_path = "Electric_Vehicle_Population_Data.csv"

# Save dataset without index column
df.to_csv(cleaned_file_path, index=False)

# Confirm save
print(f"✅ Cleaned dataset successfully saved as: {cleaned_file_path}")
```

---

### Output

```
✅ Cleaned dataset successfully saved as: Electric_Vehicle_Population_Data.csv
```

---

### Observations

- The dataset was successfully exported.
- No index column was included (index=False).
- File is now ready for use in EDA and modeling.

---

### Data Integrity Status

Data Type Consistency: ✅ CLEAN  
Missing Values: ✅ Handled  
Duplicates: ✅ None detected  
Column Naming Convention: ✅ Standardized  
Dataset Export: ✅ Completed  

---

### Why This Step Is Important

Saving the cleaned dataset:

- Improves workflow efficiency  
- Supports reproducibility  
- Allows version control  
- Prepares the dataset for modeling and deployment  

---

### Next Step

Proceed to:

- Exploratory Data Analysis (EDA)  
- Outlier Detection  
- Feature Engineering  
- Model Building  

The dataset is now fully structured, validated, and production-ready.


In [ ]:
# Step 3.0: Reload the Cleaned Dataset for EDA
# This ensures the analysis starts from the finalized, cleaned, and standardized dataset.

# Import essential libraries for data handling and visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Reload the cleaned dataset
df = pd.read_csv("Electric_Vehicle_Population_Data.csv")

# Confirm successful load
print("✅ Dataset successfully reloaded for EDA.")
print("Shape of dataset:", df.shape)

# Preview top records to verify
df.head()

## Step 3.0 — Reload the Cleaned Dataset for EDA

### Objective
To ensure that all further analysis begins from the finalized, cleaned, standardized, and validated dataset.

---

### Code Executed

```python
# Step 3.0: Reload the Cleaned Dataset for EDA

# Import essential libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Reload the cleaned dataset
df = pd.read_csv("Electric_Vehicle_Population_Data.csv")

# Confirm successful load
print("✅ Dataset successfully reloaded for EDA.")
print("Shape of dataset:", df.shape)

# Preview top records
df.head()
```

---

### Output Summary

- ✅ Dataset successfully reloaded  
- 📊 Shape of dataset: **(276,828 rows × 16 columns)**  
- 🔍 Preview confirms:
  - Standardized column names (lowercase, underscores)
  - Clean categorical formatting
  - Corrected data types
  - No duplicates
  - Missing values already handled

---

### Observations

- The dataset structure is consistent with the cleaned version saved earlier.
- Column naming convention remains standardized:
  - `vin_(1_10)`
  - `postal_code`
  - `model_year`
  - `electric_vehicle_type`
  - `clean_alternative_fuel_vehicle_(cafv)_eligibility`
  - etc.
- Data integrity checks from previous steps remain valid:
  - ✔ Missing values handled  
  - ✔ Duplicates removed  
  - ✔ Data types enforced  
  - ✔ Column names standardized  

This confirms we are starting EDA from a reliable and production-ready dataset.

---

### Why This Step Is Important

- Prevents accidental use of unclean raw data  
- Ensures reproducibility of results  
- Creates a clear separation between:
  - Data Cleaning Phase  
  - Exploratory Data Analysis Phase  
- Makes the workflow professional and portfolio-ready  

---

### Next Step

Proceed to **Step 3.1 — Exploratory Data Analysis (EDA)**:
- Descriptive statistics
- Distribution analysis
- Categorical summaries
- Visualization of key variables
- Relationship exploration

We are now officially in the EDA phase. 🚀


In [ ]:
# ============================================================
# Step 3.1: Summary Statistics (Numerical + Categorical)
# ============================================================

print("📊 DATASET SHAPE:")
print("Rows, Columns:", df.shape)
print("-" * 50)

# ---------------------------
# 1️⃣ Identify column types
# ---------------------------

num_cols = df.select_dtypes(include=["int64", "float64"]).columns
cat_cols = df.select_dtypes(include=["object"]).columns

print("🔢 Total Numerical Columns:", len(num_cols))
print("🔤 Total Categorical Columns:", len(cat_cols))
print("-" * 50)

# ---------------------------
# 2️⃣ Numerical Summary
# ---------------------------

print("📈 NUMERICAL SUMMARY STATISTICS:\n")
display(df[num_cols].describe().T)

print("-" * 50)

# ---------------------------
# 3️⃣ Categorical Summary
# ---------------------------

print("📋 CATEGORICAL SUMMARY STATISTICS:\n")
display(df[cat_cols].describe().T)

print("-" * 50)

# ---------------------------
# 4️⃣ Unique Value Counts (Categorical)
# ---------------------------

print("🔎 UNIQUE VALUE COUNTS (Categorical Columns):\n")

for col in cat_cols:
    print(f"{col}: {df[col].nunique()} unique values")

print("-" * 50)


## 📊 Step 3.1: Summary Statistics & Data Overview

This step explores the dataset’s structure, dimensions, and descriptive statistics for both numerical and categorical variables.

---

### 📦 Dataset Shape

- **Rows:** 276,828  
- **Columns:** 16  
- **Total Numerical Columns:** 6  
- **Total Categorical Columns:** 10  

This confirms we are working with a large-scale dataset suitable for meaningful analysis.

---

## 🔢 Numerical Summary Statistics

The numerical variables analyzed include:

- `postal_code`
- `model_year`
- `electric_range`
- `legislative_district`
- `dol_vehicle_id`
- `2020_census_tract`

### Key Observations:

- **Model Year**
  - Mean ≈ 2022  
  - Range: 1999 – 2026  
  - Most vehicles are recent models (2021–2024 range).

- **Electric Range**
  - Median ≈ 32 miles  
  - Wide variation (0 – 337 miles)  
  - Many vehicles have 0 range (likely PHEVs or unknown battery info).

- **Postal Code**
  - Reasonable distribution within Washington State.

- **2020 Census Tract**
  - Large numeric identifiers (should be treated as categorical/identifier, not continuous measure).

---

## 🗂️ Categorical Summary Statistics

Categorical variables analyzed include:

- `vin_(1_10)`
- `county`
- `city`
- `state`
- `make`
- `model`
- `electric_vehicle_type`
- `clean_alternative_fuel_vehicle_(cafv)_eligibility`
- `vehicle_location`
- `electric_utility`

### Key Insights:

- **State**
  - 51 unique values  
  - Dominant value: WA  

- **County**
  - 250 unique counties  
  - Most frequent: King County  

- **City**
  - 896 unique cities  
  - Seattle appears most frequently  

- **Make**
  - 47 unique manufacturers  
  - Most common: TESLA  

- **Model**
  - 185 unique models  
  - Most common: MODEL Y  

- **Electric Vehicle Type**
  - 2 categories:
    - Battery Electric Vehicle (BEV)
    - Plug-in Hybrid Electric Vehicle (PHEV)
  - BEVs dominate the dataset.

- **Census Tract & Vehicle Location**
  - High cardinality features (many unique values).

---

## 🔍 Why This Step Is Important

✔ Helps understand distribution patterns  
✔ Identifies dominant categories  
✔ Detects potential outliers  
✔ Reveals data imbalance  
✔ Guides feature engineering decisions  
✔ Supports modeling strategy  

---

### 🚀 Conclusion

The dataset is:

- Large and structured
- Mostly recent EV registrations
- Dominated by BEVs
- Heavily concentrated in Washington State
- Rich in categorical diversity (high-cardinality fields)

We are now fully prepared to proceed to:

➡ **Distribution Analysis (Univariate EDA)**
➡ **Categorical Frequency Visualizations**
➡ **Bivariate Relationship Exploration**



In [ ]:
# Step 3.3: Missing Value Analysis

# Calculate the number and percentage of missing values per column
missing_values = df.isnull().sum()
missing_percent = (missing_values / len(df)) * 100

# Combine results into a single summary DataFrame
missing_summary = pd.DataFrame({
    'Missing_Values': missing_values,
    'Percent_Missing': missing_percent.round(2)
}).sort_values(by='Percent_Missing', ascending=False)

# Display missing value summary
print("🔍 Missing Value Summary:\n")
display(missing_summary)

# Step 3.2 — Re-Validation: Missing Value Analysis Before EDA

## Objective
Before proceeding to Univariate Distribution Analysis and deeper Exploratory Data Analysis (EDA), we re-applied Missing Value Analysis to ensure no missing data reappeared after saving and reloading the cleaned dataset.

This step confirms dataset integrity and prevents silent data issues during visualization or modeling.

---

## Code

# Check total missing values per column
df.isnull().sum()

# Check total missing values in entire dataset
df.isnull().sum().sum()

---

## Output Interpretation

- All columns returned **0 missing values**
- Total dataset missing values = **0**
- Dataset shape remains consistent (276,828 rows × 16 columns)

---

## Observation

✔ No missing values reappeared  
✔ Data integrity confirmed  
✔ Cleaned dataset is stable after reload  
✔ Safe to proceed to visualization stage  

This validation ensures that:
- Histograms will not distort due to NaNs
- Bar charts will reflect accurate category frequencies
- Bivariate plots will not silently drop observations

---

## Why This Step Is Important

Even after cleaning, missing values can sometimes:
- Reappear due to incorrect export/import
- Be introduced during type conversion
- Be hidden inside categorical transformations

Re-checking protects the credibility of the analysis.

---

## Next Stage: Exploratory Data Analysis (EDA)

We will now proceed in the following structured order:

### Step 3.3 — Univariate Analysis
- Histograms (Numerical Variables)
- Boxplots (Outlier Check)
- Distribution Shape Interpretation

### Step 3.4 — Categorical Frequency Visualization
- Bar Charts
- Count Plots
- Top Categories Analysis

### Step 3.5 — Bivariate Relationship Exploration
- Scatter Plots (Numerical vs Numerical)
- Boxplots by Category
- Grouped Bar Charts
- Correlation Heatmap (after correlation matrix)

In [ ]:
# ================================
# Step 3.3 — Univariate Analysis
# ================================

import matplotlib.pyplot as plt
import seaborn as sns

# Identify numerical columns
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns

print("Numerical Columns:")
print(numerical_cols)

# -------------------------------
# 1️⃣ Histograms (Distribution)
# -------------------------------

for col in numerical_cols:
    plt.figure(figsize=(8,5))
    sns.histplot(df[col], bins=30, kde=True)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.show()

# -------------------------------
# 2️⃣ Boxplots (Outlier Check)
# -------------------------------

for col in numerical_cols:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=df[col])
    plt.title(f'Boxplot of {col}')
    plt.xlabel(col)
    plt.show()

# -------------------------------
# 3️⃣ Skewness Check
# -------------------------------

skewness = df[numerical_cols].skew().sort_values(ascending=False)
print("\nSkewness of Numerical Variables:")
print(skewness)

# Step 3.3 — Univariate Analysis (Numerical Variables)

## Objective
To analyze the distribution, spread, skewness, and potential outliers of all numerical variables in the Electric Vehicle dataset using histograms and boxplots.

---

## Variables Analyzed
- postal_code
- model_year
- electric_range
- legislative_district
- dol_vehicle_id
- 2020_census_tract

---

## Distribution Observations

### 1️⃣ postal_code
- Extremely negatively skewed (-28.10)
- Large concentration around higher values (~98,000–100,000)
- Numerous outliers detected in boxplot
- Acts more like a categorical/geographic identifier than a true numerical variable

✔ Recommendation: Treat as categorical (not for scaling or regression interpretation)

---

### 2️⃣ model_year
- Moderately left-skewed (-1.24)
- Strong concentration between 2018–2025
- Reflects surge in recent EV adoption
- Older years appear as lower-frequency outliers

✔ Insight: Dataset is dominated by recent EV registrations

---

### 3️⃣ electric_range
- Positively skewed (2.13)
- Large cluster below 50 miles
- Long right tail up to ~350 miles
- Many high-range outliers

✔ Insight: Mix of low-range older EVs and high-range modern EVs  
✔ Potential transformation candidate (log transformation if needed)

---

### 4️⃣ legislative_district
- Slight negative skew (-0.43)
- Fairly evenly distributed between 0–49
- No extreme distortion
- No significant outlier issue

✔ Insight: Balanced geographic legislative distribution

---

### 5️⃣ dol_vehicle_id
- Slight negative skew (-0.35)
- Wide numerical spread
- Contains extreme values but behaves as an identifier

✔ Recommendation: Should not be used as predictive numerical feature

---

### 6️⃣ 2020_census_tract
- Extremely negatively skewed (-25.69)
- Strong clustering near upper range
- Many extreme outliers
- Acts as geographic identifier rather than continuous measurement

✔ Recommendation: Treat as categorical/geospatial reference

---

## Outlier Summary

Boxplots reveal:
- Significant outliers in postal_code, electric_range, and 2020_census_tract
- Mild skew in model_year
- Legislative district relatively stable

Outliers in geographic/ID columns are expected and not necessarily problematic.

---

## Key Insights from Univariate Analysis

- Dataset is heavily dominated by recent EV registrations (2018–2025)
- Electric range varies widely, showing technological progression
- Geographic identifiers (postal_code, census tract, vehicle ID) show artificial skew due to coding structure
- Not all numerical columns should be treated as continuous features

---

## Feature Engineering Implications

Before modeling:
- Exclude dol_vehicle_id
- Convert postal_code and 2020_census_tract to categorical if used
- Consider transformation for electric_range if modeling regression
- Keep model_year for temporal trend analysis

---

## Next Step

Proceed to:

# Step 3.4 — Categorical Frequency Visualization
- Bar charts
- Count plots
- Dominant category analysis

EDA Progress: ✅ Univariate Complete  
Ready for Categorical Analysis 🚀

In [ ]:
# ==========================================
# Step 3.4 — Categorical Frequency Analysis
# ==========================================

import matplotlib.pyplot as plt
import seaborn as sns

# Identify categorical columns
categorical_cols = df.select_dtypes(include=['object', 'category']).columns

print("Categorical Columns:")
print(categorical_cols)

# ------------------------------------------
# 1️⃣ Frequency Table for Each Category
# ------------------------------------------

for col in categorical_cols:
    print(f"\nTop Categories in {col}:")
    print(df[col].value_counts().head(10))


# ------------------------------------------
# 2️⃣ Bar Chart / Count Plot Visualization
# ------------------------------------------

for col in categorical_cols:
    plt.figure(figsize=(8,5))
    
    # Limit to top 10 categories for readability
    top_categories = df[col].value_counts().head(10).index
    
    sns.countplot(
        data=df,
        y=col,
        order=top_categories
    )
    
    plt.title(f'Top 10 Categories in {col}')
    plt.xlabel('Count')
    plt.ylabel(col)
    plt.show()

# Step 3.4 — Categorical Frequency Visualization

## Objective
To analyze categorical variable distributions, detect dominance patterns, identify imbalance, and determine which features are suitable for modeling in the Electric Vehicle dataset.

---

## Categorical Variables Analyzed
- vin_(1_10)
- county
- city
- state
- make
- model
- electric_vehicle_type
- clean_alternative_fuel_vehicle_(cafv)_eligibility
- vehicle_location
- electric_utility

---

## Key Distribution Insights

### 1️⃣ vin_(1_10)
- High-cardinality production identifier
- Top categories show similar frequencies (~1,070–1,175)
- No meaningful business interpretation

✔ Action: Drop before modeling (identifier feature)

---

### 2️⃣ County Distribution
- King County dominates (~136k vehicles)
- Large drop to Snohomish (~34k) and Pierce (~22k)
- Strong metropolitan concentration

✔ Insight: EV adoption heavily concentrated in urban counties.

---

### 3️⃣ City Distribution
- Seattle leads (~42k)
- Bellevue, Vancouver, Redmond follow
- Clear clustering around major urban hubs

✔ Insight: Urban centers are primary drivers of EV ownership.

---

### 4️⃣ State Distribution
- Washington (WA) overwhelmingly dominant (~276k)
- Other states negligible

✔ Action: Drop (low variability, minimal predictive value)

---

### 5️⃣ Make Distribution
- TESLA dominates (~113k)
- Significant gap to Chevrolet, Nissan, Ford
- Strong brand concentration

✔ Insight: Tesla is market leader in this dataset.
✔ Important predictive feature for modeling.

---

### 6️⃣ Model Distribution
- MODEL Y (~59k) highest
- MODEL 3 (~37k) second
- Other models significantly lower

✔ Insight: Tesla models dominate EV registrations.

---

### 7️⃣ Electric Vehicle Type
- Battery Electric Vehicles (BEV) (~221k)
- Plug-in Hybrid (PHEV) (~55k)

✔ Insight: BEVs represent ~80% of vehicles.
✔ Indicates strong transition toward fully electric vehicles.

---

### 8️⃣ CAFV Eligibility
- "Eligibility unknown" most frequent (~175k)
- Eligible (~77k)
- Not eligible (~24k)

✔ Insight: Many vehicles lack confirmed battery research status.
✔ May reflect reporting or documentation gaps.

---

### 9️⃣ Vehicle Location
- Extremely high-cardinality coordinate points
- Represents spatial identifiers
- Not suitable for direct categorical encoding

✔ Action: If needed, extract latitude/longitude for spatial modeling.

---

### 🔟 Electric Utility (Updated Analysis)

- Puget Sound Energy (including Tacoma variants) dominates heavily.
- Bonneville Power Administration appears across multiple regional subdivisions.
- Clear clustering by major utility providers.
- Large gap between top 2 utilities and the rest.

✔ Insight:
EV adoption appears strongly linked to specific utility infrastructure regions.
Energy provider concentration may reflect regional EV incentive programs or infrastructure readiness.

✔ Modeling Consideration:
Keep electric_utility, but consider grouping smaller providers if encoding.

---

## Category Imbalance Overview

| Feature | Imbalance Level | Recommended Action |
|----------|----------------|-------------------|
| state | Extremely High | Drop |
| vin_(1_10) | Very High | Drop |
| vehicle_location | Very High Cardinality | Transform |
| make | High | Keep |
| model | High | Keep |
| electric_vehicle_type | Moderate | Keep |
| county / city | Moderate | Keep |
| electric_utility | Moderate-High | Keep (possibly group rare classes) |

---

## Strategic Observations

- EV ownership is geographically concentrated in Washington urban centers.
- Tesla dominates both brand and model levels.
- Fully electric vehicles far outnumber hybrids.
- Utility providers show structured clustering, suggesting infrastructure influence.
- Several categorical fields function as identifiers rather than analytical features.

---

## Feature Engineering Implications

Before modeling:

- Drop: vin_(1_10)
- Drop: state
- Transform or engineer: vehicle_location
- Keep: make, model, electric_vehicle_type
- Keep: electric_utility (consider grouping minor categories)
- Consider grouping rare categories in high-cardinality features

---

## EDA Progress

Univariate Analysis: ✅ Complete  
Categorical Frequency Analysis: ✅ Complete  

Next Step:

# Step 3.5 — Bivariate Relationship Exploration
- Numerical vs Numerical (Scatter plots)
- Categorical vs Numerical (Boxplots)
- Correlation Matrix + Heatmap

Dataset understanding is now structurally strong and modeling-ready.

In [ ]:
# ==========================================
# Step 3.5 — Bivariate Relationship Exploration
# ==========================================

import matplotlib.pyplot as plt
import seaborn as sns

# ------------------------------------------
# 0) Setup: Identify numeric & categorical
# ------------------------------------------

numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = df.select_dtypes(include=['object', 'category']).columns

print("Numerical Columns:", list(numerical_cols))
print("Categorical Columns:", list(categorical_cols))

# Use a sample for plotting to reduce RAM usage
df_plot = df.sample(frac=0.30, random_state=42)  # adjust to 0.2 or 0.4 if needed

# ==========================================
# 1) Numerical vs Numerical (Scatter Plots)
# ==========================================

# Choose key numerical features (edit if you want)
num_features = ['model_year', 'electric_range', 'legislative_district']

# Keep only those that actually exist in your dataset
num_features = [c for c in num_features if c in df.columns]

print("\nScatter plot numeric features used:", num_features)

for i in range(len(num_features)):
    for j in range(i+1, len(num_features)):
        x = num_features[i]
        y = num_features[j]
        
        plt.figure(figsize=(7,5))
        sns.scatterplot(data=df_plot, x=x, y=y, alpha=0.4)
        plt.title(f'{x} vs {y}')
        plt.show()


# ==========================================
# 2) Categorical vs Numerical (Boxplots)
# ==========================================

# Select important categorical features (edit if you want)
cat_features = ['electric_vehicle_type', 'make', 'county']

# Keep only those that exist
cat_features = [c for c in cat_features if c in df.columns]

# Numerical target for boxplots
num_target = 'electric_range' if 'electric_range' in df.columns else None

print("\nBoxplot categorical features used:", cat_features)
print("Numerical boxplot target:", num_target)

if num_target is not None:
    for cat in cat_features:
        # Limit to top 10 categories for readability
        top_cats = df[cat].value_counts().head(10).index
        
        plt.figure(figsize=(10,5))
        sns.boxplot(
            data=df_plot[df_plot[cat].isin(top_cats)],
            x=cat,
            y=num_target
        )
        plt.title(f'{num_target} by {cat} (Top 10 Categories)')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("⚠️ No suitable numerical target found for boxplots.")


# ==========================================
# 3) Correlation Matrix + Heatmap
# ==========================================

corr = df[numerical_cols].corr()

print("\nCorrelation Matrix:")
print(corr)

plt.figure(figsize=(10,6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="Blues")
plt.title("Correlation Heatmap (Numerical Features)")
plt.show()

# Step 3.5 — Bivariate Relationship Exploration

## Objective
To examine relationships between numerical features and between categorical and numerical features in order to identify trends, correlations, and potential predictive patterns.

---

# 1️⃣ Numerical vs Numerical (Scatter Plot Analysis)

### model_year vs electric_range

Observation:
- Clear clustering of low ranges (0–50 miles) across all years.
- Higher electric ranges (200–330 miles) appear mostly from 2016 onward.
- Suggests technological improvements over time.
- However, correlation matrix shows moderate negative correlation (-0.55), likely due to older plug-in hybrid vehicles having smaller electric-only range.

Insight:
EV battery capacity significantly improved in later years, but hybrid models influence the overall pattern.

---

### model_year vs legislative_district

Observation:
- No visible trend or directional pattern.
- Data points are evenly distributed across districts.

Insight:
Legislative district does not appear strongly related to vehicle model year.

---

### electric_range vs legislative_district

Observation:
- Wide distribution of electric ranges across all districts.
- No obvious district-specific concentration of higher range vehicles.

Insight:
Electric range is not geographically clustered at legislative district level.

---

# 2️⃣ Categorical vs Numerical (Boxplot Analysis)

### electric_range by electric_vehicle_type

Observation:
- Battery Electric Vehicles (BEV) show significantly higher ranges.
- Plug-in Hybrid Electric Vehicles (PHEV) concentrated mostly below 50 miles.
- BEV distribution extends above 300 miles.

Insight:
Vehicle type is a strong determinant of electric range.
This is a key predictive feature.

---

### electric_range by make

Observation:
- Tesla models dominate the higher range spectrum (200–330 miles).
- Chevrolet and Nissan show mid-to-high variability.
- Other brands cluster at lower ranges.

Insight:
Brand strongly influences electric range capability.
Make should be retained for modeling.

---

### electric_range by county

Observation:
- All counties show similar range distributions.
- No county-specific dominance in higher range vehicles.
- Outliers appear across all counties.

Insight:
Geographic county alone does not strongly determine electric range.

---

# 3️⃣ Correlation Matrix & Heatmap (Numerical Features)

Key Correlations:

- model_year vs electric_range: **-0.55**
  - Moderate relationship (affected by hybrid distribution).

- postal_code vs 2020_census_tract: **0.52**
  - Expected spatial relationship.

- model_year vs dol_vehicle_id: **0.34**
  - Administrative sequencing effect.

- electric_range vs dol_vehicle_id: **-0.19**
  - Weak relationship.

Most other correlations are near zero.

Insight:
No severe multicollinearity detected.
Numerical features are largely independent.

---

# Overall Bivariate Conclusions

- Electric range is strongly influenced by vehicle type and brand.
- Geographic variables (county, district) show limited impact on range.
- Battery technology improvement is visible over time.
- No major multicollinearity concerns in numerical data.

---

# EDA Progress

Univariate Analysis: ✅ Complete  
Categorical Frequency Analysis: ✅ Complete  
Bivariate Relationship Exploration: ✅ Complete  

Next Step:

# Step 3.6 — Correlation and Multicollinearity Analysis 
Then → Feature Engineering
Then → Modeling

In [ ]:
!pip install statsmodels

In [ ]:
# ==========================================
# Step 3.6 — Multicollinearity Analysis (VIF)
# ==========================================

import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ------------------------------------------
# 1️⃣ Select numerical columns only
# ------------------------------------------

numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
print("Numerical Columns:", list(numerical_cols))

# ------------------------------------------
# 2️⃣ Remove obvious identifier columns
# (IDs usually inflate VIF artificially)
# ------------------------------------------

cols_to_remove = ['dol_vehicle_id']  # Add others if needed
numerical_cols = [col for col in numerical_cols if col not in cols_to_remove]

print("Numerical Columns Used for VIF:", numerical_cols)

# ------------------------------------------
# 3️⃣ Create VIF DataFrame
# ------------------------------------------

X = df[numerical_cols].dropna()

vif_data = pd.DataFrame()
vif_data["Feature"] = X.columns
vif_data["VIF"] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]

# ------------------------------------------
# 4️⃣ Sort results
# ------------------------------------------

vif_data = vif_data.sort_values(by="VIF", ascending=False)

print("\nVariance Inflation Factor (VIF) Results:")
print(vif_data)

# Step 3.6 — Multicollinearity Analysis (VIF)

## Objective
To formally assess multicollinearity among numerical features using Variance Inflation Factor (VIF) before proceeding to feature engineering and modeling.

---

## Numerical Features Evaluated

- postal_code
- model_year
- electric_range
- legislative_district
- 2020_census_tract

(dol_vehicle_id was removed as an identifier)

---

## VIF Results

| Feature | VIF |
|----------|------|
| 2020_census_tract | **1410.88** |
| postal_code | 1.37 |
| legislative_district | 1.00 |
| electric_range | 1.00 |
| model_year | 0.00 |

---

## Interpretation

### 1️⃣ Severe Multicollinearity Detected

- **2020_census_tract has extremely high VIF (~1411).**
- This indicates that it is almost perfectly predictable from other spatial variables.
- Likely highly correlated with postal_code and/or geographic identifiers.

✔ This feature introduces severe redundancy.

---

### 2️⃣ Other Features

- postal_code → VIF = 1.37 (Low)
- legislative_district → VIF ≈ 1.00 (No multicollinearity)
- electric_range → VIF ≈ 1.00 (Independent)
- model_year → VIF ≈ 0 (No multicollinearity concern)

✔ These variables are safe to retain.

---

## Conclusion

- There is **no widespread multicollinearity problem**.
- Only **2020_census_tract** shows severe inflation.
- This is expected for geographic encoding variables.

---

## Recommended Action

Drop:
- 2020_census_tract (redundant geographic identifier)

Keep:
- postal_code
- model_year
- electric_range
- legislative_district

---

## EDA Progress

Univariate Analysis: ✅ Complete  
Categorical Frequency Analysis: ✅ Complete  
Bivariate Analysis: ✅ Complete  
Correlation Analysis: ✅ Complete  
Multicollinearity (VIF): ✅ Complete  

Dataset is now structurally clean and ready for:

# Step 4 — Feature Engineering

In [ ]:
# ==========================================
# Step 4.1 — Drop Redundant Features
# ==========================================

features_to_drop = [
    'vin_(1_10)',
    'state',
    'vehicle_location',
    '2020_census_tract',
    'dol_vehicle_id'
]

# Keep only columns that exist
features_to_drop = [col for col in features_to_drop if col in df.columns]

df = df.drop(columns=features_to_drop)

print("Dropped Features:", features_to_drop)
print("Remaining Shape:", df.shape)

## Step 4.1 — Drop Redundant Features

### Objective
At this stage, I removed redundant and non-informative features to improve model performance and prevent data leakage or multicolarity issues.

---

### Features Dropped
- `vin_(1_10)` → Vehicle identifier (no predictive value)
- `state` → Low variance (mostly one state)
- `vehicle_location` → High-cardinality coordinate data
- `2020_census_tract` → Extremely high VIF (1410+) indicating severe multicollinearity
- `dol_vehicle_id` → Unique identifier (no predictive value)

---

### Code
```python
features_to_drop = [
    'vin_(1_10)',
    'state',
    'vehicle_location',
    '2020_census_tract',
    'dol_vehicle_id'
]

# Keep only columns that exist
features_to_drop = [col for col in features_to_drop if col in df.columns]

df = df.drop(columns=features_to_drop)

print("Dropped Features:", features_to_drop)
print("Remaining Shape:", df.shape)
```

---

### Output
```
Dropped Features: ['vin_(1_10)', 'state', 'vehicle_location', '2020_census_tract', 'dol_vehicle_id']
Remaining Shape: (276828, 11)
```

---

### Observations
- The dataset now contains **276,828 rows and 11 columns**
- All identifier and high-collinearity variables have been removed
- The dataset structure is now cleaner and more modeling-ready

---

### Why This Step Is Important
Removing redundant features:
- Prevents multicollinearity problems
- Reduces noise
- Improves model interpretability
- Speeds up training
- Prevents data leakage

---

### Next Step
Proceed to **Step 4.2 — Create Derived Features (e.g., vehicle_age)** to enhance predictive power.

In [ ]:
# ==========================================
# Step 4.2 — Create Vehicle Age Feature
# ==========================================

current_year = 2026  # adjust if needed
df['vehicle_age'] = current_year - df['model_year']

print(df[['model_year', 'vehicle_age']].head())

## Step 4.2 — Create Derived Feature: Vehicle Age

### Objective
To improve model interpretability and predictive power, I created a new feature called `vehicle_age` from the existing `model_year` column.

Instead of using raw model year values, vehicle age provides a more meaningful representation of how old each vehicle is, which can directly impact electric range performance.

---

### Code
```python
# ==========================================
# Step 4.2 — Create Vehicle Age Feature
# ==========================================

current_year = 2026  # adjust if needed
df['vehicle_age'] = current_year - df['model_year']

print(df[['model_year', 'vehicle_age']].head())
```

---

### Output
```
   model_year  vehicle_age
0        2018            8
1        2016           10
2        2023            3
3        2025            1
4        2016           10
```

---

### Observations
- Vehicles from recent years have lower `vehicle_age` values.
- Older vehicles correctly show higher age values.
- The transformation is mathematically consistent and logically correct.

---

### Why This Step Is Important
Creating `vehicle_age`:
- Improves interpretability compared to raw year values
- Helps capture potential battery degradation effects
- Reduces reliance on absolute year scale
- Often performs better in regression models
- Enhances model generalization

---

### Dataset Status
The dataset now includes an engineered numerical feature ready for modeling.

---

### Next Step
Proceed to **Step 4.3 — Encoding Categorical Variables** to convert text features into model-compatible numerical format.

In [ ]:
# Separate features

numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = df.select_dtypes(include=['object', 'category']).columns

print("Numerical:", list(numerical_cols))
print("Categorical:", list(categorical_cols))

## Step 4.3.1 — Separate Numerical & Categorical Features

### Objective
Before applying encoding techniques, I separated the dataset into numerical and categorical columns.  
This step ensures that only categorical variables are encoded, while numerical features remain unchanged.

---

### Code
```python
# Separate features
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = df.select_dtypes(include=['object', 'category']).columns

print("Numerical:", list(numerical_cols))
print("Categorical:", list(categorical_cols))
```

---

### Output
```
Numerical: ['postal_code', 'model_year', 'electric_range', 
            'legislative_district', 'vehicle_age']

Categorical: ['county', 'city', 'make', 'model', 
              'electric_vehicle_type', 
              'clean_alternative_fuel_vehicle_(cafv)_eligibility', 
              'electric_utility']
```

---

### Observations
- The dataset contains both structured numerical and categorical variables.
- Numerical features will be used directly in modeling.
- Categorical features require transformation into numeric format before training machine learning models.

---

### Why This Step Is Important
- Prevents accidental encoding of numeric variables.
- Ensures clean and controlled preprocessing.
- Maintains modeling integrity.
- Prepares the dataset for systematic encoding.

---

### Next Step
Proceed to **Step 4.3.2 — Apply One-Hot Encoding to Categorical Variables** to convert text-based features into model-compatible numerical format.

In [ ]:
# ==========================================
# Step 4.3.2 — One-Hot Encoding
# ==========================================

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print("Encoded Shape:", df_encoded.shape)

## Step 4.3.2 — Apply One-Hot Encoding to Categorical Variables

### Objective
At this stage, I converted categorical variables into numerical format using One-Hot Encoding.  
Machine learning models cannot interpret text-based features directly, so this transformation is necessary for modeling.

---

### Code
```python
# ==========================================
# Step 4.3.2 — One-Hot Encoding
# ==========================================

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print("Encoded Shape:", df_encoded.shape)
```

---

### Output
```
Encoded Shape: (276828, 1458)
```

---

### Observations
- The dataset still contains **276,828 rows**, meaning no records were lost.
- The number of columns increased significantly (from 11 to 1458).
- This increase is due to high-cardinality categorical features (e.g., city, model, electric_utility).

---

### Why This Step Is Important
- Converts categorical variables into model-compatible numeric format.
- Prevents ordinal bias (unlike label encoding).
- Preserves information across all categories.
- Prepares the dataset for regression modeling.

---

### Important Note
The high number of features (1458) may:
- Increase training time
- Increase risk of overfitting
- Require feature selection or dimensionality reduction later

---

### Dataset Status
The dataset is now fully numeric and ready for:
- Train-Test Split
- Model Training
- Feature Importance Analysis

---

### Next Step
Proceed to **Step 4.4 — Train-Test Split** to prepare the dataset for regression modeling.

In [ ]:
# ==========================================
# Step 4.4 — Train-Test Split
# ==========================================

from sklearn.model_selection import train_test_split

# Define target variable (Regression)
y = df_encoded['electric_range']

# Define feature matrix
X = df_encoded.drop(columns=['electric_range'])

print("Feature Shape:", X.shape)
print("Target Shape:", y.shape)

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("\nTrain/Test Split Shapes:")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

## Step 4.4 — Train-Test Split

### Objective
To properly evaluate model performance, I split the dataset into training and testing sets.

This ensures:
- The model learns patterns from training data.
- Performance is evaluated on unseen data.
- Overfitting risk is reduced.

---

### Code
```python
# Define feature matrix
X = df_encoded.drop(columns=['electric_range'])

print("Feature Shape:", X.shape)
print("Target Shape:", y.shape)

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("\nTrain/Test Split Shapes:")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)
```

---

### Output
```
Feature Shape: (276828, 1457)
Target Shape: (276828,)

Train/Test Split Shapes:
X_train: (221462, 1457)
X_test: (55366, 1457)
y_train: (221462,)
y_test: (55366,)
```

---

### Observations
- The dataset contains **276,828 records**.
- 80% (221,462 rows) used for training.
- 20% (55,366 rows) reserved for testing.
- 1,457 features used for modeling.
- The target variable (`electric_range`) was properly separated to prevent data leakage.

---

### Why This Step Is Important
- Ensures unbiased evaluation.
- Prevents training and testing on the same data.
- Allows proper performance measurement using R² and MSE.
- Maintains modeling integrity.

---

### Dataset Status
The dataset is now fully numeric and properly split, ready for regression modeling.

---

### Next Step
Proceed to **Step 4.5 — Train Baseline Regression Model (Linear Regression)**.

In [ ]:
# ==========================================
# Step 4.5 — Baseline Linear Regression
# ==========================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Initialize model
lr_model = LinearRegression()

# Train model
lr_model.fit(X_train, y_train)

# Make predictions
y_pred = lr_model.predict(X_test)

# Evaluate model
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("Linear Regression Results:")
print("MSE:", mse)
print("RMSE:", rmse)
print("R² Score:", r2)

## Step 4.5 — Baseline Model: Linear Regression

### Objective
I trained a baseline Linear Regression model to predict `electric_range`.

This model serves as a reference point for evaluating more advanced models later.

---

### Code
```python
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Initialize model
lr_model = LinearRegression()

# Train model
lr_model.fit(X_train, y_train)

# Make predictions
y_pred = lr_model.predict(X_test)

# Evaluate model
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("Linear Regression Results:")
print("MSE:", mse)
print("RMSE:", rmse)
print("R² Score:", r2)
```

---

### Output
```
Linear Regression Results:
MSE: 358.7879
RMSE: 18.9417
R² Score: 0.9417
```

---

### Observations
- The model explains approximately **94.17% of the variance** in electric range.
- RMSE of ~18.94 indicates the average prediction error is about 19 miles.
- The performance is strong for a baseline linear model.
- No severe underfitting is observed.

---

### Interpretation
- The engineered features (especially `vehicle_age`) and encoded categorical variables contributed significantly to model performance.
- High R² suggests strong linear relationships within the dataset.
- This provides a solid benchmark for comparison with more complex models.

---

### Why This Step Is Important
- Establishes a baseline performance level.
- Allows performance comparison with Random Forest and Gradient Boosting.
- Helps detect whether complex models truly add value.

---

### Model Status
Baseline regression model successfully trained and evaluated.

---

### Next Step
Proceed to **Step 4.6 — Train Advanced Models (Random Forest Regressor)** to compare performance improvements.

In [ ]:
# ==========================================
# Step 4.6 — Random Forest Regressor
# ==========================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Initialize model
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train model
rf_model.fit(X_train, y_train)

# Make predictions
y_pred_rf = rf_model.predict(X_test)

# Evaluate model
mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest Results:")
print("MSE:", mse_rf)
print("RMSE:", rmse_rf)
print("R² Score:", r2_rf)

# Step 4.6 — Random Forest Regressor

## Objective
In this step, I trained a Random Forest Regressor to improve predictive performance compared to the baseline Linear Regression model.

---

## Model Configuration

I initialized the model with:

- n_estimators = 100 (number of decision trees)
- random_state = 42 (for reproducibility)
- n_jobs = -1 (to utilize all available CPU cores)

This ensures consistent and efficient training.

---

## Model Training Process

- I trained the model using `X_train` and `y_train`.
- I generated predictions using `X_test`.
- I evaluated performance using MSE, RMSE, and R² score.

---

## Model Performance Results

- **Mean Squared Error (MSE):** 13.49  
- **Root Mean Squared Error (RMSE):** 3.67  
- **R² Score:** 0.9978  

---

## Interpretation

Compared to the Linear Regression model (R² ≈ 0.94), the Random Forest model shows a substantial improvement in predictive accuracy.

Key observations:

- The extremely high R² score (0.9978) indicates that the model explains nearly all variance in the target variable (`electric_range`).
- The low RMSE (3.67) shows predictions are very close to actual values.
- Random Forest captures complex non-linear relationships and feature interactions better than linear models.

---

## Model Comparison Insight

| Model               | R² Score | RMSE |
|---------------------|----------|------|
| Linear Regression   | ~0.94    | ~18.94 |
| Random Forest       | 0.9978   | 3.67 |

The Random Forest significantly outperforms the linear model.

---

## Professional Reflection

The very high R² suggests excellent predictive power.  
However, performance this strong requires careful validation to ensure:

- No data leakage
- No overfitting
- Proper feature engineering

Further validation and model comparison will confirm generalization strength.

---

## Next Step

Proceed to:
- Train Gradient Boosting Regressor  
or  
- Perform model comparison and validation analysis

In [ ]:
# ============================================
# Step 4.7 — Gradient Boosting Regressor
# ============================================

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# 1) Initialize model
gbr_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

# 2) Train model
gbr_model.fit(X_train, y_train)

# 3) Make predictions
y_pred_gbr = gbr_model.predict(X_test)

# 4) Evaluate model
mse_gbr = mean_squared_error(y_test, y_pred_gbr)
rmse_gbr = np.sqrt(mse_gbr)
r2_gbr = r2_score(y_test, y_pred_gbr)

print("Gradient Boosting Regressor Results:")
print("MSE:", mse_gbr)
print("RMSE:", rmse_gbr)
print("R² Score:", r2_gbr)

# Step 4.7 — Gradient Boosting Regressor

## Objective
In this step, I trained a Gradient Boosting Regressor to further evaluate advanced ensemble performance and compare it with both Linear Regression and Random Forest models.

---

## Model Configuration

The model was initialized with the following parameters:

- n_estimators = 200  
- learning_rate = 0.05  
- max_depth = 3  
- random_state = 42  

Gradient Boosting builds trees sequentially, where each tree corrects the errors of the previous one.

---

## Model Training Process

- I trained the model using `X_train` and `y_train`.
- I generated predictions using `X_test`.
- I evaluated performance using MSE, RMSE, and R² score.

---

## Model Performance Results

- **Mean Squared Error (MSE):** 46.98  
- **Root Mean Squared Error (RMSE):** 6.85  
- **R² Score:** 0.9924  

---

## Interpretation

The Gradient Boosting model achieved strong predictive performance:

- The R² score (0.9924) indicates that the model explains over 99% of the variance in electric range.
- The RMSE (6.85) shows predictions are still very close to actual values.
- Performance is better than Linear Regression (R² ≈ 0.94).
- However, it does not outperform the Random Forest model (R² ≈ 0.9978).

---

## Model Comparison Summary

| Model                 | R² Score | RMSE  |
|-----------------------|----------|-------|
| Linear Regression     | ~0.94    | ~18.94 |
| Random Forest         | 0.9978   | 3.67  |
| Gradient Boosting     | 0.9924   | 6.85  |

Random Forest remains the strongest performer so far.

---

## Professional Reflection

Gradient Boosting performs extremely well and captures complex relationships effectively.  
However, Random Forest achieved the best overall accuracy on this dataset.

The next step is to:

- Perform a structured model comparison analysis  
- Check for overfitting  
- Analyze feature importance  
- Or move toward model validation and deployment readiness

# Step 4.8 — Model Comparison Summary

## Objective
In this step, I compared the performance of all trained regression models to determine which model best predicts electric range.

The models evaluated were:

- Linear Regression  
- Random Forest Regressor  
- Gradient Boosting Regressor  

---

## Performance Comparison

| Model                 | MSE      | RMSE   | R² Score |
|-----------------------|----------|--------|----------|
| Linear Regression     | 358.79   | 18.94  | 0.9417   |
| Random Forest         | 13.49    | 3.67   | 0.9978   |
| Gradient Boosting     | 46.98    | 6.85   | 0.9924   |

---

## Performance Interpretation

### 1️⃣ Linear Regression
- Serves as a baseline model.
- Captures linear relationships only.
- Lowest performance among the three models.
- RMSE is significantly higher compared to ensemble methods.

### 2️⃣ Random Forest Regressor
- Best overall performer.
- Highest R² score (0.9978).
- Lowest RMSE (3.67).
- Effectively captures complex non-linear relationships.
- Strong ability to model interactions between encoded features.

### 3️⃣ Gradient Boosting Regressor
- Very strong performance (R² = 0.9924).
- More controlled sequential learning approach.
- Slightly less accurate than Random Forest on this dataset.
- Still significantly better than Linear Regression.

---

## Key Insights

- Ensemble tree-based models drastically outperform Linear Regression.
- The relationship between features and electric range is clearly non-linear.
- Random Forest demonstrates superior generalization on the test set.
- Extremely high R² values require careful validation to rule out overfitting.

---

## Final Model Selection

Based on evaluation metrics:

🏆 **Random Forest Regressor is currently the best-performing model.**

It will be selected for:

- Feature importance analysis  
- Model validation  
- Potential deployment  

---

## Next Step

Proceed to:

- Step 4.9 — Feature Importance Analysis (from Random Forest)

This will help interpret which variables most strongly influence electric range predictions.

In [ ]:
# ============================================
# Step 4.9 — Feature Importance (Random Forest)
# ============================================

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Get feature importances
importances = rf_model.feature_importances_

# Create DataFrame
feature_importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": importances
})

# Sort by importance
feature_importance_df = feature_importance_df.sort_values(
    by="Importance",
    ascending=False
)

# Display top 15 features
print("Top 15 Most Important Features:")
print(feature_importance_df.head(15))

# Plot top 15
top_features = feature_importance_df.head(15)

plt.figure(figsize=(10,6))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.gca().invert_yaxis()
plt.title("Top 15 Feature Importances (Random Forest)")
plt.xlabel("Importance Score")
plt.show()

# Step 4.9 — Feature Importance Analysis (Random Forest)

## Objective
In this step, I analyzed feature importance using the trained Random Forest model to understand which variables most strongly influence electric range predictions.

Random Forest provides built-in feature importance scores based on how much each feature reduces prediction error across trees.

---

## Top 15 Most Important Features

The most influential features identified were:

1. **vehicle_age** — 0.3418  
2. **model_year** — 0.2952  
3. **electric_vehicle_type_Plug-in Hybrid Electric Vehicle (PHEV)** — 0.2351  
4. make_NISSAN  
5. model_LEAF  
6. clean_alternative_fuel_vehicle_(cafv)_eligibility  
7. make_TESLA  
8. model_BOLT EV  
9. make_BMW  
10. model_E-GOLF  
11. model_I3  
12. model_SOUL  
13. model_MODEL 3  
14. make_VOLKSWAGEN  

---

## Interpretation of Results

### 1️⃣ vehicle_age (Most Important Feature)
- This confirms that vehicle age is the strongest predictor of electric range.
- Older vehicles likely have lower range due to battery degradation.
- The engineered feature (vehicle_age) significantly improved model interpretability.

### 2️⃣ model_year
- Strong predictive influence.
- Closely related to vehicle_age.
- Indicates technological improvements in newer vehicles.

### 3️⃣ Electric Vehicle Type (PHEV)
- Vehicle type strongly affects range capacity.
- Plug-in Hybrid vehicles behave differently from full Battery Electric Vehicles.

### 4️⃣ Make & Model Variables
Several encoded categorical variables (Nissan, Tesla, BMW, Bolt EV, Leaf, etc.) appear in the top features.

This indicates:
- Different manufacturers and models have significantly different battery capacities.
- Brand-level and model-level engineering differences strongly influence electric range.

---

## Key Insight

The dominance of:
- vehicle_age  
- model_year  
- vehicle type  
- specific models  

confirms that electric range is heavily influenced by:

- Technological generation
- Battery design differences
- Manufacturer-specific engineering

This validates the use of tree-based ensemble models.

---

## Professional Reflection

The feature importance results align with domain knowledge:

- Battery degradation over time affects range.
- Newer vehicle models have improved battery efficiency.
- Different brands and models offer different performance levels.

The engineered feature (vehicle_age) successfully captured temporal effects and became the most predictive variable.

---

## Next Step

Proceed to:

- Step 4.10 — Overfitting & Generalization Check  
or  
- Step 4.11 — Model Validation & Deployment Readiness  
or  
- Save and Export the Best Model (Random Forest)

In [ ]:
# ============================================
# Step 4.10 — Overfitting Check (Random Forest)
# ============================================

from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# Predict on training set
y_train_pred_rf = rf_model.predict(X_train)

# Training performance
train_mse = mean_squared_error(y_train, y_train_pred_rf)
train_rmse = np.sqrt(train_mse)
train_r2 = r2_score(y_train, y_train_pred_rf)

# Test performance (already computed, but recompute for clarity)
y_test_pred_rf = rf_model.predict(X_test)

test_mse = mean_squared_error(y_test, y_test_pred_rf)
test_rmse = np.sqrt(test_mse)
test_r2 = r2_score(y_test, y_test_pred_rf)

print("Random Forest Overfitting Check:")
print("\nTraining Performance:")
print("MSE:", train_mse)
print("RMSE:", train_rmse)
print("R²:", train_r2)

print("\nTest Performance:")
print("MSE:", test_mse)
print("RMSE:", test_rmse)
print("R²:", test_r2)

# Step 4.10 — Overfitting & Generalization Check (Random Forest)

## Objective
In this step, I evaluated whether the Random Forest model is overfitting or generalizing well by comparing training and test performance.

Overfitting occurs when a model performs significantly better on training data than on unseen test data.

---

## Training vs Test Performance

### 🔹 Training Performance
- **MSE:** 8.05  
- **RMSE:** 2.84  
- **R² Score:** 0.9987  

### 🔹 Test Performance
- **MSE:** 13.49  
- **RMSE:** 3.67  
- **R² Score:** 0.9978  

---

## Performance Gap Analysis

- Training R² = 0.9987  
- Test R² = 0.9978  
- Difference = **0.0009**

This gap is extremely small.

---

## Interpretation

The Random Forest model shows:

- Very high performance on both training and test sets.
- Minimal performance gap between training and testing.
- No evidence of severe overfitting.
- Strong generalization ability.

The slight difference in RMSE (2.84 vs 3.67) is expected and healthy.

This suggests that the model is learning meaningful patterns rather than memorizing the training data.

---

## Why Generalization Is Strong

This strong generalization likely results from:

- A large dataset (276k+ records)
- Structured and meaningful predictors (vehicle_age, model_year, vehicle type, manufacturer)
- Tree-based ensemble robustness
- Proper train/test splitting

---

## Professional Conclusion

The Random Forest model demonstrates:

✔ Excellent predictive accuracy  
✔ Strong generalization  
✔ No significant overfitting  
✔ Stable performance across datasets  

At this stage, the model is considered production-ready from a performance perspective.

---

## Next Step

Proceed to:

- Step 4.11 — Model Validation & Deployment Readiness  
or  
- Save and Export the Best Model (Random Forest)

In [ ]:
# ============================================================
# Step 4.11 (Sample-Based) — Model Validation & Readiness
# Uses a sample of the full dataset to reduce RAM/time
# ============================================================

import numpy as np
import pandas as pd
import joblib
from datetime import datetime

from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import mean_squared_error, r2_score

# ------------------------------------------------------------
# 4.11.S1 — Create a sample for validation (recommended)
# ------------------------------------------------------------
# Assumes you already have full X and y defined.
# If you ONLY have X_train/y_train, you can sample from those instead.

SAMPLE_FRAC = 0.20   # 20% sample (change to 0.10 or 0.05 if needed)
RANDOM_STATE = 42

# Sample rows (keeps X and y aligned)
idx_sample = X.sample(frac=SAMPLE_FRAC, random_state=RANDOM_STATE).index
X_s = X.loc[idx_sample].copy()
y_s = y.loc[idx_sample].copy()

print("Sample shapes:")
print("X_s:", X_s.shape)
print("y_s:", y_s.shape)

# ------------------------------------------------------------
# 4.11.S2 — Cross-Validation on the sample (Random Forest)
# ------------------------------------------------------------
# Assumes rf_model already exists (your best RandomForestRegressor)

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "r2": "r2",
    "neg_mse": "neg_mean_squared_error",
    "neg_rmse": "neg_root_mean_squared_error"
}

cv_results = cross_validate(
    rf_model,
    X_s, y_s,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=True
)

cv_summary_sample = pd.DataFrame({
    "Metric": ["R2", "MSE", "RMSE"],
    "Train_Mean": [
        cv_results["train_r2"].mean(),
        (-cv_results["train_neg_mse"]).mean(),
        (-cv_results["train_neg_rmse"]).mean()
    ],
    "Train_Std": [
        cv_results["train_r2"].std(),
        (-cv_results["train_neg_mse"]).std(),
        (-cv_results["train_neg_rmse"]).std()
    ],
    "Test_Mean": [
        cv_results["test_r2"].mean(),
        (-cv_results["test_neg_mse"]).mean(),
        (-cv_results["test_neg_rmse"]).mean()
    ],
    "Test_Std": [
        cv_results["test_r2"].std(),
        (-cv_results["test_neg_mse"]).std(),
        (-cv_results["test_neg_rmse"]).std()
    ]
})

print("\nStep 4.11.S2 — Sample-Based CV Summary (5-Fold):")
display(cv_summary_sample)

# ------------------------------------------------------------
# 4.11.S3 — Optional: quick diagnostic on your existing test set
# ------------------------------------------------------------
# If you already have X_test and y_test, keep using them (no extra RAM).
# Otherwise, you can skip this section.

y_test_pred = rf_model.predict(X_test)
residuals = y_test - y_test_pred

test_mse = mean_squared_error(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)
test_r2 = r2_score(y_test, y_test_pred)

print("\nStep 4.11.S3 — Test Metrics (Full test set):")
print("MSE:", test_mse)
print("RMSE:", test_rmse)
print("R²:", test_r2)

# Light residual plots (sample points only)
import matplotlib.pyplot as plt

N_PLOT = 5000  # reduce if needed
plt.figure(figsize=(8, 5))
plt.scatter(y_test_pred[:N_PLOT], residuals[:N_PLOT])
plt.axhline(0)
plt.title("Residuals vs Predictions (Sample)")
plt.xlabel("Predicted")
plt.ylabel("Residual (Actual - Predicted)")
plt.show()

plt.figure(figsize=(8, 5))
plt.hist(residuals[:N_PLOT], bins=40)
plt.title("Residual Distribution (Sample)")
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.show()

# ------------------------------------------------------------
# 4.11.S4 — Save model + metadata (include sample CV summary)
# ------------------------------------------------------------
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

model_path = f"best_random_forest_model_{timestamp}.pkl"
meta_path = f"model_metadata_{timestamp}.pkl"

joblib.dump(rf_model, model_path)

metadata = {
    "model_type": "RandomForestRegressor",
    "saved_at": timestamp,
    "validation_mode": f"sample_based_cv_frac_{SAMPLE_FRAC}",
    "sample_shape": X_s.shape,
    "feature_count": X.shape[1],
    "feature_names": list(X.columns),
    "cv_summary_sample": cv_summary_sample,
    "test_metrics": {"mse": test_mse, "rmse": test_rmse, "r2": test_r2}
}

joblib.dump(metadata, meta_path)

print("\nSaved Files:")
print("Model:", model_path)
print("Metadata:", meta_path)

# Step 4.11 — Model Validation & Deployment Readiness (Sample-Based)

## Objective
In this step, I performed sample-based cross-validation and residual diagnostics to further validate the robustness and deployment readiness of the Random Forest model.

Due to dataset size (276k+ records), I used a 20% sample for efficient validation.

---

## Sample Overview

- Sample Size: 55,366 records  
- Feature Count: 1,457  

This sample maintains dataset structure while reducing computational load.

---

## 5-Fold Cross-Validation Results (Sample-Based)

| Metric | Train Mean | Test Mean | Train Std | Test Std |
|--------|------------|-----------|-----------|----------|
| R²     | 0.9991     | 0.9975    | 0.00003   | 0.00018  |
| MSE    | 5.55       | 15.52     | 0.21      | 1.31     |
| RMSE   | 2.36       | 3.94      | 0.04      | 0.17     |

---

## Cross-Validation Interpretation

- Training and validation R² values are extremely close.
- Standard deviations are very small.
- Performance is stable across folds.
- No evidence of instability or variance spikes.

This confirms strong generalization across different subsets of the data.

---

## Full Test Set Performance (Hold-Out Validation)

- **MSE:** 13.49  
- **RMSE:** 3.67  
- **R² Score:** 0.9978  

The hold-out test results align closely with cross-validation results.

---

## Residual Analysis

### Residuals vs Predictions
- Residuals are centered around zero.
- No clear systematic pattern observed.
- Minor variance increase at very high predicted values (expected in real-world data).

### Residual Distribution
- Highly concentrated around zero.
- Few extreme residuals on both positive and negative sides.
- Distribution suggests strong predictive accuracy.

---

## Overfitting Assessment

- Training R² ≈ 0.9987  
- Test R² ≈ 0.9978  
- Cross-Validation R² ≈ 0.9975  

The performance gap is extremely small.

There is no significant overfitting.

---

## Deployment Readiness

The Random Forest model demonstrates:

✔ High predictive accuracy  
✔ Stable cross-validation performance  
✔ Minimal variance across folds  
✔ Strong generalization  
✔ Clean residual behavior  

The model is considered deployment-ready.

---

## Model Artifacts Saved

- Random Forest Model (`.pkl`)
- Model Metadata (`.pkl`)
- Feature names
- Cross-validation summary
- Test metrics

---

## Professional Conclusion

The Random Forest model has been:

- Trained  
- Evaluated  
- Compared  
- Interpreted  
- Validated  
- Stress-tested  

This completes the full predictive modeling lifecycle for this project.

# Final Project Conclusion

In this project, I implemented a complete end-to-end data science workflow to predict electric vehicle range.

The process included structured data cleaning, exploratory data analysis, multicollinearity assessment, feature engineering, model comparison, overfitting diagnostics, and deployment preparation.

After evaluating multiple regression models, the Random Forest Regressor demonstrated the best balance between predictive accuracy and generalization performance.

The final model was serialized and deployed using Streamlit, transforming the analytical workflow into an interactive prediction application.

This project demonstrates not only model-building capability, but also disciplined validation and production-readiness preparation.